Übung: kind
-----------

![](https://kind.sigs.k8s.io/logo/logo.png)

Quelle: https://kind.sigs.k8s.io/

- - -

Kind (Kubernetes IN Docker) ist ein leistungsstarkes Werkzeug, das Entwicklern und Administratoren hilft, Kubernetes-Cluster auf einfache und schnelle Weise lokal zu erstellen und zu verwalten. 

Kind wurde speziell entwickelt, um Kubernetes-Cluster in Docker-Containern zu betreiben, was es ideal für Entwicklungs-, Test- und CI/CD-Umgebungen macht.

* Einfache Installation und Nutzung: Da Kind Docker-Container verwendet, kann ein Kubernetes-Cluster mit nur wenigen Befehlen eingerichtet werden. Dies ermöglicht eine schnelle Bereitstellung und einen unkomplizierten Start.

* Portabilität: Durch die Nutzung von Docker-Containern sind die erstellten Kubernetes-Cluster plattformunabhängig und können auf verschiedenen Betriebssystemen wie Linux, macOS und Windows ausgeführt werden.

* Schnelle Iteration: Kind ermöglicht schnelle Iterationen bei der Entwicklung und beim Testen von Kubernetes-Anwendungen. Cluster können in Minuten erstellt und gelöscht werden, was eine flexible und dynamische Entwicklungsumgebung unterstützt.

* CI/CD-Integration: Kind eignet sich hervorragend für die Integration in Continuous Integration und Continuous Deployment (CI/CD)-Pipelines. Entwickler können ihre Anwendungen in einer realistischen Kubernetes-Umgebung testen, bevor sie in die Produktion überführt werden.

* Leichtgewichtig: Im Gegensatz zu herkömmlichen Kubernetes-Installationen benötigt Kind weniger Ressourcen, da es auf Docker-Containern basiert. Dies macht es zu einer idealen Wahl für Entwickler, die auf Laptops oder in begrenzten Entwicklungsumgebungen arbeiten.

- - -

In der Übung erstellen wir einen Kubernetes Cluster in einem Container. 

Dazu benötigen wir zuerst eine Konfigurationsdatei welche die Anzahl control und Worker Nodes festlegt. Hier ein control und drei Worker Nodes


In [ ]:
%%bash
cat <<%EOF% >kind-config.yaml
# kind-config.yaml
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
  - role: control-plane
  - role: worker
  - role: worker
%EOF%

Mit dieser Konfiguration können wir einen Kubernetes Cluster erstellen, wo jede Node als Container läuft.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/docker-kind
kind create cluster --config kind-config.yaml --name docker-kind

Der Kubernetes Cluster muss als Docker Container sichtbar und via kubectl ansprechbar sein:

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/docker-kind
docker stats --no-stream docker-kind-control-plane docker-kind-worker docker-kind-worker2
kubectl get nodes

Intressant sind die gestartet Kubernetes Dienste, welche bei `kind` den Kubernetes Standards entsprechen

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/docker-kind
kubectl get pods -A

- - -

### Vergleich mit der aktuellen Kubernetes Distribution

Zum Vergleich die Kubernetes Dienste von der aktuellen Kubernetes Distribution.

In [ ]:
%%bash
sudo kubectl get pods -n kube-system
pstree -npTA

Wir stellen fest, dass sich die Kubernetes Distributionen hauptsächlich 
* in der Art der verwendeteten Produkte (z.B. für Netzwerk kindnet vs. calico, Key-Value Server etcd vs. dqlite)
* ob die System Prozesse als Container oder Linux System Dienste laufen
  
unterscheiden.

- - -

### WebShop

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/docker-kind
kubectl create namespace ms-rest
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/catalog-deployment.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/customer-deployment.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/order-deployment.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-deployment/webshop-deployment.yaml 
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml
kubectl apply --namespace ms-rest -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/webshop-service.yaml
kubectl get pod,services --namespace ms-rest

**Portweiterleitung (Port Forwarding)**

Ein wichtiger Aspekt bei der Arbeit mit Kubernetes ist die Fähigkeit, auf die im Cluster ausgeführten Dienste von aussen zuzugreifen. 

Mit einer einfach Portweiterleitung (Port Forwarding) können wir auf unseren Webshop zugreifen.

Mittels des **Stop Button** (oben) können wir die Weiterleitung stoppen.

In [ ]:
%%bash
export KUBECONFIG=$HOME/data/.kube/docker-kind
echo http://"$(cat ~/data/server-ip)":8888/webshop
kubectl port-forward --namespace ms-rest --address 0.0.0.0 service/webshop 8888:8080

- - -

### Aufräumen 

Dazu löschen wir den `kind` Kubernetes Cluster

In [ ]:
%%bash
kind delete cluster --name docker-kind